# Qwen3-TTS (L4 / vLLM-Omni 2-Stage CustomVoice, VoiceDesign & 3s Voice Clone)

Serves `Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice`, `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign`, and `Qwen/Qwen3-TTS-12Hz-1.7B-Base` on a Colab L4 GPU via `vllm serve --omni` (Stage 0 Talker + Stage 1 Code2Wav 12Hz Tokenizer) plus an auxiliary multi-task GPU worker, supporting 9 built-in premium timbres (`Vivian`, `Serena`, `Uncle_Fu`, `Dylan`, `Eric`, `Ryan`, `Aiden`, `Ono_Anna`, `Sohee`), natural-language instruction control, VoiceDesign persona creation, 3-second zero-shot voice cloning, Voice-Design-then-Clone workflow, 10 languages, and Google Drive voice/output persistence.

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "qwen3-tts")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8000"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = "/content/ct"                                            # logs + binaries of this notebook
os.makedirs(WORK, exist_ok=True)
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to /content/ct/<name>.log."""
    with open(f"{WORK}/{name}.log", "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        if heartbeat is not None:
            try:
                heartbeat(int(time.monotonic() - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- install vLLM-Omni + Qwen3-TTS & pre-stage 12Hz models on local SSD
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

VARIANT_REPOS = {
    "customvoice": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    "1.7b-customvoice": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    "voicedesign": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    "1.7b-voicedesign": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "1.7b-base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "0.6b-customvoice": "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
    "0.6b-base": "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
}
VARIANT = os.environ.get("CT_QWEN3_TTS_VARIANT", "customvoice").strip().lower()
MODEL_REPO = os.environ.get("CT_QWEN3_TTS_MODEL", "").strip() or VARIANT_REPOS.get(VARIANT, VARIANT_REPOS["customvoice"])
MODEL_ALIAS = os.environ.get("CT_QWEN3_TTS_ALIAS", "qwen3-tts").strip()
PRELOAD_ALL = os.environ.get("CT_QWEN3_TTS_PRELOAD_ALL", "1").strip()
STAGE0_GPU_MEM = float(os.environ.get("CT_QWEN3_TTS_GPU_MEM_STAGE0", "0.30"))
STAGE1_GPU_MEM = float(os.environ.get("CT_QWEN3_TTS_GPU_MEM_STAGE1", "0.20"))
MAX_MODEL_LEN = int(os.environ.get("CT_QWEN3_TTS_MAX_MODEL_LEN", "4096"))
MAX_NUM_SEQS = int(os.environ.get("CT_QWEN3_TTS_MAX_NUM_SEQS", "16"))
AUX_PORT = int(os.environ.get("CT_QWEN3_TTS_AUX_PORT", "8001"))

QT_WORK = Path("/content/qwen3-tts")
VENV_DIR = QT_WORK / "venv"
AUX_VENV_DIR = QT_WORK / "aux-venv"
MODELS_ROOT = QT_WORK / "model"
MODEL_DIR = MODELS_ROOT / MODEL_REPO.split("/")[-1]
SPEAKERS_DIR = QT_WORK / "speakers"
DRIVE_VOICES_DIR = Path(DATA_DIR) / "voices"
DRIVE_OUTPUT_DIR = Path(DATA_DIR) / "outputs"
DEFAULT_REF_WAV = QT_WORK / "default_ref.wav"
DEFAULT_REF_TEXT = (
    "Okay. Yeah. I resent you. I love you. I respect you. "
    "But you know what? You blew it! And thanks to you."
)

# Two-tier JIT kernel precompilation cache (Local NVMe SSD <-> Google Drive)
FLASHINFER_DRIVE_DIR = Path(DATA_DIR) / "flashinfer-cache"
FLASHINFER_CACHE_DIR = QT_WORK / "flashinfer-cache"
TRITON_DRIVE_DIR = Path(DATA_DIR) / "triton-cache"
TRITON_CACHE_DIR = QT_WORK / "triton-cache"
VLLM_DRIVE_DIR = Path(DATA_DIR) / "vllm-cache"
VLLM_CACHE_DIR = QT_WORK / "vllm-cache"

for d in (
    QT_WORK, MODELS_ROOT, SPEAKERS_DIR, DRIVE_VOICES_DIR, DRIVE_OUTPUT_DIR,
    FLASHINFER_DRIVE_DIR, FLASHINFER_CACHE_DIR,
    TRITON_DRIVE_DIR, TRITON_CACHE_DIR,
    VLLM_DRIVE_DIR, VLLM_CACHE_DIR,
):
    d.mkdir(parents=True, exist_ok=True)

def qlog(msg):
    print(time.strftime("[%F %T] [qwen3-tts] ") + msg, flush=True)

# One archive per cache/environment/GPU avoids Drive's small-file latency.
(QT_WORK / "colab_cache.py").write_text('"""Stage optional compiler caches using one Drive archive per environment/GPU."""\nimport hashlib\nimport os\nimport shutil\nimport tarfile\nimport tempfile\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef _progress(log, message, operation):\n    stopped = threading.Event()\n    started = time.monotonic()\n\n    def heartbeat():\n        while not stopped.wait(10):\n            log(f\'{message}, elapsed {time.monotonic()-started:.0f} s\')\n\n    log(message)\n    thread = threading.Thread(target=heartbeat, daemon=True)\n    thread.start()\n    try:\n        return operation()\n    finally:\n        stopped.set()\n        thread.join()\n\n\ndef _files(root):\n    return sorted(p for p in root.rglob(\'*\') if p.is_file() and not p.is_symlink()\n                  and not p.name.endswith((\'.lock\', \'.tmp\')))\n\n\ndef _fingerprint(root, files):\n    digest = hashlib.sha256()\n    for path in files:\n        stat = path.stat()\n        digest.update(f\'{path.relative_to(root)}:{stat.st_size}:{stat.st_mtime_ns}\\n\'.encode())\n    return digest.hexdigest()\n\n\ndef _unpack(archive, local):\n    # Validate the whole archive before extracting; compiler caches contain\n    # regular files/directories only. Never follow archive or local symlinks.\n    with tarfile.open(archive, \'r:\') as tar:\n        members = tar.getmembers()\n        root = local.resolve()\n        for member in members:\n            destination = local / member.name\n            if (not (member.isfile() or member.isdir()) or Path(member.name).is_absolute()\n                    or not destination.resolve().is_relative_to(root)):\n                raise ValueError(\'Cache archive contains unsafe paths or file types\')\n            current = destination\n            while current != local:\n                if current.is_symlink():\n                    raise ValueError(\'Cache target contains a symbolic link\')\n                current = current.parent\n        tar.extractall(local, members=members, filter=\'data\')\n\n\ndef restore_cache(local, legacy, archive_dir, key, log=print):\n    local, legacy, archive_dir = map(Path, (local, legacy, archive_dir))\n    local.mkdir(parents=True, exist_ok=True)\n    archive = archive_dir / key / (local.name + \'.tar\')\n    stamp = local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')\n    if archive.is_file():\n        stat = archive.stat()\n        signature = f\'{stat.st_size}:{stat.st_mtime_ns}\'\n        if stamp.exists() and stamp.read_text() == signature and _files(local):\n            log(f\'{local.name}: already restored locally, skipping duplicate transfer\')\n            return \'local\'\n        try:\n            with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-restore-\') as tmp:\n                staged = Path(tmp) / \'cache.tar\'\n                _progress(log, f\'{local.name}: reading archive from Drive ({stat.st_size/1e6:.1f} MB)\',\n                          lambda: shutil.copyfile(archive, staged))\n                _progress(log, f\'{local.name}: unpacking on local SSD\', lambda: _unpack(staged, local))\n            stamp.write_text(signature)\n            (local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')).write_text(\n                _fingerprint(local, _files(local)))\n            log(f\'{local.name}: archive restore complete\')\n            return \'archive\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: archive unavailable, falling back to the legacy cache ({error})\')\n    if legacy.is_dir():\n        try:\n            def migrate():\n                shutil.copytree(legacy, local, dirs_exist_ok=True, symlinks=True)\n            _progress(log, f\'{local.name}: first-time migration of the legacy Drive cache (slow when there are many small files)\', migrate)\n            if _files(local):\n                save_cache(local, archive_dir, key, log)\n                return \'legacy\'\n        except (OSError, ValueError, tarfile.TarError) as error:\n            log(f\'{local.name}: legacy cache migration failed, falling back to on-demand generation ({error})\')\n    log(f\'{local.name}: no persistent cache; the service will generate it on demand\')\n    return \'miss\'\n\n\ndef save_cache(local, archive_dir, key, log=print):\n    local, archive_dir = Path(local), Path(archive_dir)\n    files = _files(local)\n    if not files:\n        log(f\'{local.name}: no cache files, skipping archive\')\n        return False\n    fingerprint = _fingerprint(local, files)\n    saved = local.parent / (\'.\' + local.name + \'-\' + key + \'.saved\')\n    destination = archive_dir / key / (local.name + \'.tar\')\n    if saved.exists() and saved.read_text() == fingerprint and destination.is_file():\n        log(f\'{local.name}: content unchanged, skipping upload\')\n        return False\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \'.\' + uuid.uuid4().hex + \'.tmp\')\n    try:\n        with tempfile.TemporaryDirectory(dir=local.parent, prefix=\'.cache-save-\') as tmp:\n            staged = Path(tmp) / \'cache.tar\'\n            def pack():\n                with tarfile.open(staged, \'w:\') as tar:\n                    for path in files:\n                        tar.add(path, arcname=str(path.relative_to(local)), recursive=False)\n            _progress(log, f\'{local.name}: packing {len(files)} files locally\', pack)\n            _progress(log, f\'{local.name}: saving archive to Drive ({staged.stat().st_size/1e6:.1f} MB)\',\n                      lambda: shutil.copyfile(staged, temporary))\n            os.replace(temporary, destination)\n        saved.write_text(fingerprint)\n        stat = destination.stat()\n        (local.parent / (\'.\' + local.name + \'-\' + key + \'.restored\')).write_text(\n            f\'{stat.st_size}:{stat.st_mtime_ns}\')\n        log(f\'{local.name}: Drive archive updated\')\n        return True\n    finally:\n        temporary.unlink(missing_ok=True)\n', encoding="utf-8")
sys.path.insert(0, str(QT_WORK))
from colab_cache import restore_cache, save_cache
_gpu_arch = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True).strip().splitlines()[0]
CACHE_KEY = "vllm-0.32.0-py312-sm" + _gpu_arch.replace(".", "")
CACHE_ARCHIVES = Path(DATA_DIR) / "cache-archives"
CACHE_PAIRS = (
    (FLASHINFER_CACHE_DIR, FLASHINFER_DRIVE_DIR),
    (TRITON_CACHE_DIR, TRITON_DRIVE_DIR),
    (VLLM_CACHE_DIR, VLLM_DRIVE_DIR),
)
for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        restore_cache(_local_c, _legacy_c, CACHE_ARCHIVES, CACHE_KEY, qlog)
    except (OSError, ValueError) as err:
        qlog(f"Cache restore warning ({_local_c.name}): {err}; continuing with local cache")
qlog("Restoring registered voice samples from Drive")
subprocess.run(["cp", "-rn", f"{DRIVE_VOICES_DIR}/.", f"{SPEAKERS_DIR}/"], check=False)
qlog("Cache/voice restore finished; preparing environment and model")

def run_logged(cmd, log_path, env=None, cwd=None, check=True):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n$ {' '.join(str(x) for x in cmd)}\n")
        f.flush()
        res = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=cwd, text=True)
    if check and res.returncode != 0:
        tail = "".join(open(log_path, errors="replace").readlines()[-40:])
        raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(str(x) for x in cmd[:4])}\n{tail}")
    return res

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], text=True
).strip()
qlog(f"GPU detected: {gpu_info} | repo={MODEL_REPO} variant={VARIANT} stage0_mem={STAGE0_GPU_MEM} stage1_mem={STAGE1_GPU_MEM}")

if not shutil.which("ffmpeg") or not shutil.which("sox"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg sox libsox-fmt-all >/dev/null 2>&1", shell=True, check=False)

if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)

def prepare_env():
    env_log = QT_WORK / "env-setup.log"
    marker = VENV_DIR / ".ct_qwen3_tts_env_ready"
    vllm_bin = VENV_DIR / "bin" / "vllm"
    py_bin = VENV_DIR / "bin" / "python"
    aux_py_bin = AUX_VENV_DIR / "bin" / "python"
    if marker.is_file() and vllm_bin.is_file() and py_bin.is_file() and aux_py_bin.is_file():
        qlog("Reusing existing vLLM-Omni + Qwen3-TTS virtualenvs")
        return VENV_DIR

    qlog(f"Installing vLLM + vLLM-Omni + Qwen3-TTS environments (log: {env_log})...")
    uv_bin = shutil.which("uv") or "uv"
    run_logged([uv_bin, "venv", "--python", "3.12", str(VENV_DIR), "--clear"], env_log)
    run_logged(
        [
            uv_bin, "pip", "install", "--python", str(py_bin),
            "vllm==0.32.0", "vllm-omni==0.32.0",
            "tiktoken", "soundfile", "huggingface_hub", "hf_xet",
        ],
        env_log,
    )

    # Lightweight auxiliary venv (reusing system PyTorch) for cross-task Qwen3TTSModel fallback
    run_logged([uv_bin, "venv", "--python", "3.12", "--system-site-packages", str(AUX_VENV_DIR), "--clear"], env_log)
    run_logged(
        [
            uv_bin, "pip", "install", "--python", str(aux_py_bin),
            "transformers==4.57.3", "accelerate==1.12.0", "qwen-tts",
            "soundfile", "librosa", "einops", "onnxruntime", "sox",
        ],
        env_log,
    )
    marker.write_text("ok\n", encoding="utf-8")
    qlog("vLLM-Omni + Qwen3-TTS virtualenvs ready")
    return VENV_DIR

def _cleanup_hf_temp():
    for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):
        shutil.rmtree(p, ignore_errors=True)
    for mp in MODELS_ROOT.glob("*/.cache"):
        shutil.rmtree(mp, ignore_errors=True)

def _ensure_default_ref_wav():
    if DEFAULT_REF_WAV.is_file() and DEFAULT_REF_WAV.stat().st_size > 1024:
        return
    drive_copy = DRIVE_VOICES_DIR / "default_ref.wav"
    if drive_copy.is_file() and drive_copy.stat().st_size > 1024:
        shutil.copyfile(drive_copy, DEFAULT_REF_WAV)
        return
    for ref_url in (
        "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/clone.wav",
        "https://raw.githubusercontent.com/vllm-project/vllm-omni/main/tests/assets/indextts2/ref_audio.wav",
    ):
        try:
            with urllib.request.urlopen(ref_url, timeout=60) as _src, open(DEFAULT_REF_WAV, "wb") as _dst:
                shutil.copyfileobj(_src, _dst)
            if DEFAULT_REF_WAV.is_file() and DEFAULT_REF_WAV.stat().st_size > 1024:
                break
        except Exception:
            pass
    if not DEFAULT_REF_WAV.is_file() or DEFAULT_REF_WAV.stat().st_size <= 1024:
        import math, struct, wave
        sr = 24000
        n = sr * 3
        with wave.open(str(DEFAULT_REF_WAV), "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sr)
            samples = [int(12000 * math.sin(2 * math.pi * 220 * i / sr)) for i in range(n)]
            wf.writeframes(struct.pack(f"<{n}h", *samples))
    if not drive_copy.is_file():
        shutil.copyfile(DEFAULT_REF_WAV, drive_copy)

def prepare_model():
    dl_log = QT_WORK / "model-download.log"
    _ensure_default_ref_wav()
    repos_to_fetch = [MODEL_REPO]
    if PRELOAD_ALL == "1":
        for extra_repo in (
            "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
            "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
            "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        ):
            if extra_repo not in repos_to_fetch:
                repos_to_fetch.append(extra_repo)

    def _is_repo_ready(r_id: str) -> bool:
        d = MODELS_ROOT / r_id.split("/")[-1]
        return (
            (d / ".ct_model_ready").is_file()
            and (d / "config.json").is_file()
            and (d / "model.safetensors").is_file()
            and (d / "speech_tokenizer" / "model.safetensors").is_file()
        )

    missing_repos = [r for r in repos_to_fetch if not _is_repo_ready(r)]
    if not missing_repos:
        qlog(f"Reusing local SSD Qwen3-TTS models in {MODELS_ROOT}: {[r.split('/')[-1] for r in repos_to_fetch]}")
        return MODEL_DIR

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"], check=True)
    _cleanup_hf_temp()
    free_gb = shutil.disk_usage("/content").free / 1e9
    qlog(f"Downloading {missing_repos} -> local SSD {MODELS_ROOT} (free SSD: {free_gb:.1f} GB; log: {dl_log})...")
    dl_env = {**os.environ, "HF_XET_CHUNK_CACHE_SIZE_BYTES": "0", "HF_HUB_ENABLE_HF_TRANSFER": "0"}
    dl_env.pop("HF_XET_HIGH_PERFORMANCE", None)
    dl_script = r"""
import shutil, sys
from pathlib import Path
from huggingface_hub import snapshot_download

models_root = Path(sys.argv[1])
repos = sys.argv[2:]

def clean():
    for p in ("/root/.cache/huggingface/xet", "/root/.cache/huggingface/hub"):
        shutil.rmtree(p, ignore_errors=True)
    for mp in models_root.glob("*/.cache"):
        shutil.rmtree(mp, ignore_errors=True)

for idx, repo_id in enumerate(repos, 1):
    target_dir = models_root / repo_id.split("/")[-1]
    target_dir.mkdir(parents=True, exist_ok=True)
    print(f"[dl] {idx}/{len(repos)} Downloading {repo_id} -> {target_dir}", flush=True)
    snapshot_download(repo_id=repo_id, local_dir=str(target_dir), max_workers=4)
    clean()
    if (target_dir / "model.safetensors").is_file() and (target_dir / "speech_tokenizer" / "model.safetensors").is_file():
        (target_dir / ".ct_model_ready").write_text(repo_id + "\n", encoding="utf-8")
"""
    try:
        run_logged([sys.executable, "-c", dl_script, str(MODELS_ROOT), *missing_repos], dl_log, env=dl_env)
    finally:
        _cleanup_hf_temp()

    if not _is_repo_ready(MODEL_REPO):
        raise RuntimeError(f"Primary model download incomplete in {MODEL_DIR}")
    total_gb = sum(p.stat().st_size for p in MODELS_ROOT.rglob("*") if p.is_file()) / 1e9
    qlog(f"Qwen3-TTS models ready on local SSD: {MODELS_ROOT} ({total_gb:.2f} GB total)")
    return MODEL_DIR

with ThreadPoolExecutor(max_workers=2) as pool:
    f_env = pool.submit(prepare_env)
    f_mod = pool.submit(prepare_model)
    while not (f_env.done() and f_mod.done()):
        for fut in (f_env, f_mod):
            if fut.done() and fut.exception():
                raise fut.exception()
        free_gb = shutil.disk_usage("/content").free / 1e9
        qlog(f"Setup heartbeat | env={'done' if f_env.done() else 'running'} "
             f"model={'done' if f_mod.done() else 'running'} | free_ssd={free_gb:.1f} GB")
        time.sleep(30)
    VENV_DIR = f_env.result()
    MODEL_DIR = f_mod.result()

qlog(f"Setup complete: venv={VENV_DIR} model={MODEL_DIR} drive_data={DATA_DIR}")


In [ ]:
# @title 2. Service -- vLLM-Omni Qwen3-TTS on 127.0.0.1:8000 + Aux Multi-Task Worker on :8001 + Bearer Auth
DEPLOY_YAML = QT_WORK / "qwen3_tts.yaml"
DEPLOY_YAML.write_text(f"""pipeline: qwen3_tts
model_runner: v2
async_chunk: true

connectors:
  connector_of_shared_memory:
    name: SharedMemoryConnector
    extra:
      talker_first_audio: true
      codec_streaming: true
      connector_get_sleep_s: 0.01
      connector_get_max_wait_first_chunk: 3000
      connector_get_max_wait: 300
      codec_chunk_frames: 25
      codec_left_context_frames: 72
      initial_codec_chunk_frames: 1
      decode_batch_max_size: 4

stages:
  - stage_id: 0
    devices: "0"
    max_num_seqs: {MAX_NUM_SEQS}
    gpu_memory_utilization: {STAGE0_GPU_MEM}
    trust_remote_code: true
    enable_prefix_caching: false
    async_scheduling: true
    max_num_batched_tokens: 512
    max_model_len: {MAX_MODEL_LEN}
    output_connectors:
      to_stage_1: connector_of_shared_memory
    default_sampling_params:
      temperature: 0.9
      top_k: 50
      max_tokens: {MAX_MODEL_LEN}
      min_tokens: 2
      repetition_penalty: 1.05
    silence_ban_frames: 0
    subtalker_sampling_params:
      do_sample: true
      temperature: 0.9
      top_k: 50
      top_p: 1.0

  - stage_id: 1
    devices: "0"
    dtype: bfloat16
    max_num_seqs: {MAX_NUM_SEQS}
    gpu_memory_utilization: {STAGE1_GPU_MEM}
    enforce_eager: false
    trust_remote_code: true
    enable_prefix_caching: false
    async_scheduling: true
    max_num_batched_tokens: 65536
    max_model_len: 65536
    input_connectors:
      from_stage_0: connector_of_shared_memory
    default_sampling_params:
      temperature: 0.0
      top_p: 1.0
      top_k: -1
      max_tokens: 65536
      repetition_penalty: 1.0
""", encoding="utf-8")

# Auxiliary multi-task worker on 127.0.0.1:8001 so CustomVoice, VoiceDesign, and Base (Voice Clone)
# all work seamlessly regardless of which checkpoint is loaded as the primary vLLM-Omni engine.
(QT_WORK / "ct_qwen3_aux_worker.py").write_text('"""Auxiliary loopback multi-task GPU worker for Qwen3-TTS (CustomVoice / VoiceDesign / Base)."""\nfrom __future__ import annotations\n\nimport gc\nimport io\nimport json\nimport os\nimport subprocess\nimport threading\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom huggingface_hub import snapshot_download\nfrom qwen_tts import Qwen3TTSModel\n\nMODELS_ROOT = Path(os.environ.get("CT_MODELS_ROOT", "/content/qwen3-tts/model"))\nAUX_PORT = int(os.environ.get("CT_QWEN3_TTS_AUX_PORT", "8001"))\n\nTASK_REPOS = {\n    "CustomVoice": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",\n    "VoiceDesign": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",\n    "Base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",\n}\n\n_LOCK = threading.Lock()\n_LOADED_TASK: str | None = None\n_LOADED_MODEL: Qwen3TTSModel | None = None\n\n\ndef _get_or_load_model(task_type: str) -> Qwen3TTSModel:\n    global _LOADED_TASK, _LOADED_MODEL\n    if _LOADED_TASK == task_type and _LOADED_MODEL is not None:\n        return _LOADED_MODEL\n    if _LOADED_MODEL is not None:\n        del _LOADED_MODEL\n        _LOADED_MODEL = None\n        _LOADED_TASK = None\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    repo_id = TASK_REPOS.get(task_type, TASK_REPOS["Base"])\n    local_dir = MODELS_ROOT / repo_id.split("/")[-1]\n    if not (local_dir / "model.safetensors").is_file():\n        local_dir.mkdir(parents=True, exist_ok=True)\n        snapshot_download(repo_id=repo_id, local_dir=str(local_dir), max_workers=4)\n\n    _LOADED_MODEL = Qwen3TTSModel.from_pretrained(\n        str(local_dir),\n        device_map="cuda:0",\n        dtype=torch.bfloat16,\n    )\n    _LOADED_TASK = task_type\n    return _LOADED_MODEL\n\n\ndef _encode_audio(wav: np.ndarray, sr: int, fmt: str = "wav", speed: float = 1.0) -> tuple[bytes, int]:\n    wav_f32 = np.asarray(wav, dtype=np.float32).reshape(-1)\n    buf = io.BytesIO()\n    sf.write(buf, wav_f32, sr, format="WAV", subtype="PCM_16")\n    raw_wav = buf.getvalue()\n    fmt = (fmt or "wav").lower()\n    if abs(speed - 1.0) < 1e-3 and fmt == "wav":\n        return raw_wav, sr\n\n    af_filters = []\n    rem = max(0.25, min(4.0, float(speed)))\n    while rem > 2.0 + 1e-3:\n        af_filters.append("atempo=2.0")\n        rem /= 2.0\n    while rem < 0.5 - 1e-3:\n        af_filters.append("atempo=0.5")\n        rem /= 0.5\n    if abs(rem - 1.0) >= 1e-3:\n        af_filters.append(f"atempo={rem:.4f}")\n\n    cmd = ["ffmpeg", "-nostdin", "-v", "error", "-i", "pipe:0"]\n    if af_filters:\n        cmd += ["-filter:a", ",".join(af_filters)]\n    if fmt == "mp3":\n        cmd += ["-f", "mp3", "pipe:1"]\n    elif fmt == "flac":\n        cmd += ["-f", "flac", "pipe:1"]\n    elif fmt == "opus":\n        cmd += ["-c:a", "libopus", "-f", "ogg", "pipe:1"]\n    elif fmt == "pcm":\n        cmd += ["-f", "s16le", "-acodec", "pcm_s16le", "pipe:1"]\n    else:\n        cmd += ["-f", "wav", "-acodec", "pcm_s16le", "pipe:1"]\n    res = subprocess.run(cmd, input=raw_wav, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=120, check=False)\n    if res.returncode == 0 and res.stdout:\n        return res.stdout, sr\n    return raw_wav, sr\n\n\nclass AuxHandler(BaseHTTPRequestHandler):\n    def log_message(self, fmt: str, *args: Any) -> None:\n        pass\n\n    def _send_json(self, status: int, data: dict[str, Any]) -> None:\n        body = json.dumps(data, ensure_ascii=False).encode("utf-8")\n        self.send_response(status)\n        self.send_header("Content-Type", "application/json")\n        self.send_header("Content-Length", str(len(body)))\n        self.end_headers()\n        self.wfile.write(body)\n\n    def do_GET(self) -> None:\n        if self.path == "/health":\n            self._send_json(200, {"status": "ok", "loaded_task": _LOADED_TASK})\n        else:\n            self._send_json(404, {"error": "not found"})\n\n    def do_POST(self) -> None:\n        if self.path != "/synthesize":\n            self._send_json(404, {"error": "not found"})\n            return\n        length = int(self.headers.get("Content-Length") or "0")\n        raw = self.rfile.read(length)\n        try:\n            req = json.loads(raw)\n            task_type = str(req.get("task_type") or "Base")\n            text = str(req.get("input") or req.get("text") or "").strip()\n            language = str(req.get("language") or "Auto")\n            instruct = str(req.get("instructions") or req.get("instruct") or "").strip()\n            speaker = str(req.get("voice") or req.get("speaker") or "vivian").strip()\n            ref_audio = req.get("ref_audio")\n            if isinstance(ref_audio, str) and ref_audio.startswith("file://"):\n                ref_audio = ref_audio[len("file://"):]\n            ref_text = req.get("ref_text")\n            x_vec_only = bool(req.get("x_vector_only_mode", not bool(ref_text and str(ref_text).strip())))\n            fmt = str(req.get("response_format") or "wav").lower()\n            speed = float(req.get("speed") or 1.0)\n            max_new_tokens = int(req.get("max_new_tokens") or 2048)\n\n            with _LOCK:\n                model = _get_or_load_model(task_type)\n                if task_type == "VoiceDesign":\n                    wavs, sr = model.generate_voice_design(\n                        text=text,\n                        language=language,\n                        instruct=instruct or "Natural, expressive human voice.",\n                        max_new_tokens=max_new_tokens,\n                    )\n                elif task_type == "CustomVoice":\n                    wavs, sr = model.generate_custom_voice(\n                        text=text,\n                        language=language,\n                        speaker=speaker,\n                        instruct=instruct,\n                        max_new_tokens=max_new_tokens,\n                    )\n                else:\n                    wavs, sr = model.generate_voice_clone(\n                        text=text,\n                        language=language,\n                        ref_audio=ref_audio,\n                        ref_text=ref_text if not x_vec_only else None,\n                        x_vector_only_mode=x_vec_only,\n                        max_new_tokens=max_new_tokens,\n                    )\n            audio_bytes, out_sr = _encode_audio(wavs[0], int(sr), fmt=fmt, speed=speed)\n            mime_map = {\n                "wav": "audio/wav",\n                "mp3": "audio/mpeg",\n                "flac": "audio/flac",\n                "opus": "audio/ogg",\n                "pcm": "audio/pcm",\n            }\n            self.send_response(200)\n            self.send_header("Content-Type", mime_map.get(fmt, "audio/wav"))\n            self.send_header("Content-Length", str(len(audio_bytes)))\n            self.send_header("X-Audio-Sample-Rate", str(out_sr))\n            self.send_header("X-Qwen3-TTS-Engine", f"aux-{task_type}")\n            self.end_headers()\n            self.wfile.write(audio_bytes)\n        except Exception as e:\n            import traceback\n            self._send_json(500, {"error": {"message": str(e), "traceback": traceback.format_exc()[-800:]}})\n\n\nif __name__ == "__main__":\n    srv = ThreadingHTTPServer(("127.0.0.1", AUX_PORT), AuxHandler)\n    print(f"[aux-worker] listening on 127.0.0.1:{AUX_PORT}", flush=True)\n    srv.serve_forever()\n', encoding="utf-8")

(QT_WORK / "colab_asgi_common.py").write_text('"""Shared security, Drive path containment, and audio helpers for Colab ASGI middlewares."""\nfrom __future__ import annotations\n\nimport base64\nimport glob as glob_mod\nimport os\nimport re\nimport secrets\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nAUDIO_EXTS = (\n    ".wav", ".flac", ".mp3", ".ogg", ".m4a", ".aac", ".opus", ".webm", ".mp4", ".mkv", ".wma", ".mpga", ".mpeg",\n)\nDEFAULT_SKIP_DIRS = {\n    ".git", "node_modules", "__pycache__", ".venv", "venv", "aux-venv", "custom_nodes",\n    "models", ".runtimes", "Takeout", ".Trash", ".ipynb_checkpoints",\n    "flashinfer-cache", "triton-cache", "vllm-cache", "cache-archives", "kernel-archives",\n}\nVOICE_NAME_RE = re.compile(r"^[a-z0-9][a-z0-9._-]{0,63}$")\n\n\ndef validate_voice_name(name: str) -> str:\n    """Validate and normalize a custom voice identifier against path traversal."""\n    vname = (name or "").strip().lower()\n    if not vname or ".." in vname or not VOICE_NAME_RE.fullmatch(vname):\n        raise ValueError(f"Invalid voice name \'{name}\': use 1-64 lowercase [a-z0-9._-] characters without \'..\'")\n    return vname\n\n\ndef _get_header(headers: Any, key: str) -> str:\n    val = headers.get(key)\n    if val:\n        return str(val)\n    if isinstance(headers, dict):\n        low = key.lower()\n        for k, v in headers.items():\n            if str(k).lower() == low and v:\n                return str(v)\n    return ""\n\n\ndef verify_bearer_auth(headers: Any, api_token: str) -> bool:\n    """Fail-closed constant-time Bearer / X-API-Key authentication check."""\n    if not api_token:\n        return False\n    expected_bearer = f"Bearer {api_token}"\n    auth = _get_header(headers, "authorization")\n    if auth and secrets.compare_digest(auth.encode("utf-8"), expected_bearer.encode("utf-8")):\n        return True\n    xkey = _get_header(headers, "x-api-key")\n    if xkey and secrets.compare_digest(xkey.encode("utf-8"), api_token.encode("utf-8")):\n        return True\n    return False\n\n\ndef sanitize_header_param(value: str) -> str:\n    """Strip CR/LF/NUL and escape quotes/backslashes for Content-Disposition parameters."""\n    cleaned = (value or "").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n    return cleaned.replace("\\\\", "\\\\\\\\").replace(\'"\', \'\\\\"\')\n\n\ndef build_multipart_form(\n    fields: list[tuple[str, str]],\n    files: list[tuple[str, str, str, bytes]] | None = None,\n) -> tuple[str, bytes]:\n    """Build a safe multipart/form-data body returning (content_type, body_bytes)."""\n    boundary = f"----ctform{secrets.token_hex(16)}"\n    parts: list[bytes] = []\n    for key, val in fields:\n        safe_key = sanitize_header_param(str(key))\n        parts.append(\n            f\'--{boundary}\\r\\nContent-Disposition: form-data; name="{safe_key}"\\r\\n\\r\\n{val}\\r\\n\'.encode("utf-8")\n        )\n    for field_name, filename, content_type, data in files or []:\n        safe_field = sanitize_header_param(str(field_name))\n        safe_filename = sanitize_header_param(str(filename))\n        safe_ct = (content_type or "application/octet-stream").replace("\\r", "").replace("\\n", "").replace("\\x00", "")\n        parts.append(\n            (\n                f"--{boundary}\\r\\n"\n                f\'Content-Disposition: form-data; name="{safe_field}"; filename="{safe_filename}"\\r\\n\'\n                f"Content-Type: {safe_ct}\\r\\n\\r\\n"\n            ).encode("utf-8")\n            + data\n            + b"\\r\\n"\n        )\n    parts.append(f"--{boundary}--\\r\\n".encode("utf-8"))\n    return f"multipart/form-data; boundary={boundary}", b"".join(parts)\n\n\ndef allowed_drive_roots(\n    data_dir: Path,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n) -> tuple[Path, ...]:\n    """Return resolved root directories that Drive/data operations may access."""\n    roots: list[Path] = []\n    if mydrive_root.is_dir():\n        roots.append(mydrive_root.resolve())\n    roots.append(data_dir.resolve())\n    return tuple(dict.fromkeys(roots))\n\n\ndef _ensure_within_roots(candidate: Path, roots: tuple[Path, ...], raw_spec: str) -> Path:\n    resolved = candidate.resolve()\n    if not any(resolved.is_relative_to(root) for root in roots):\n        raise ValueError(f"Path \'{raw_spec}\' escapes allowed Drive/data directory")\n    return resolved\n\n\ndef resolve_drive_path(\n    raw_spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> Path:\n    """Resolve a Drive or service-data path strictly inside allowed roots."""\n    if "\\x00" in (raw_spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    spec = (raw_spec or "").strip()\n\n    is_file_scheme = spec.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if spec.startswith(prefix):\n            spec = spec[len(prefix):]\n            break\n\n    if is_file_scheme:\n        candidate = Path("/" + spec.lstrip("/"))\n        return _ensure_within_roots(candidate, roots, raw_spec)\n\n    had_leading_slash = spec.startswith("/")\n    clean = spec.lstrip("/")\n    if not clean:\n        if default_subdir and (data_dir / default_subdir).is_dir():\n            return _ensure_within_roots(data_dir / default_subdir, roots, raw_spec)\n        return roots[0]\n\n    if had_leading_slash and not clean.startswith(("content/drive/", "drive/", "MyDrive/", "MyDrive")):\n        # Allow explicit absolute path ONLY if it already lies inside an allowed root (e.g. DATA_DIR in --no-drive mode)\n        return _ensure_within_roots(Path("/" + clean), roots, raw_spec)\n\n    candidates: list[Path] = []\n    if clean.startswith("content/drive/"):\n        candidates.append(Path("/") / clean)\n    elif clean.startswith("drive/"):\n        candidates.append(drive_root.parent / clean)\n    elif clean.startswith("MyDrive/"):\n        candidates.append(drive_root / clean)\n    elif clean == "MyDrive":\n        candidates.append(mydrive_root)\n    elif clean.startswith("colab-llm/") and mydrive_root.is_dir():\n        candidates.append(mydrive_root / clean)\n        candidates.append(data_dir / clean)\n    else:\n        first_seg = clean.split("/", 1)[0]\n        already_prefixed = bool(\n            default_subdir\n            and (\n                clean == default_subdir\n                or clean.startswith(f"{default_subdir}/")\n                or (first_seg != clean and (data_dir / first_seg).is_dir())\n            )\n        )\n        if default_subdir and not already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        candidates.append(data_dir / clean)\n        if default_subdir and already_prefixed:\n            candidates.append(data_dir / default_subdir / clean)\n        if mydrive_root.is_dir():\n            candidates.append(mydrive_root / clean)\n\n    checked: list[Path] = []\n    for cand in candidates:\n        resolved = _ensure_within_roots(cand, roots, raw_spec)\n        checked.append(resolved)\n        if resolved.exists():\n            return resolved\n    if not checked:\n        raise ValueError(f"Path \'{raw_spec}\' could not be resolved inside allowed Drive/data directory")\n    return checked[0]\n\n\ndef rel_to_mydrive(p: Path, mydrive_root: Path = Path("/content/drive/MyDrive")) -> str:\n    try:\n        return str(p.resolve().relative_to(mydrive_root.resolve()))\n    except Exception:\n        return str(p)\n\n\ndef list_drive_audio_files(\n    spec: str,\n    *,\n    data_dir: Path,\n    default_subdir: str | None = None,\n    recursive: bool = True,\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n    skip_dirs: set[str] | None = None,\n) -> list[dict[str, Any]]:\n    """List audio files matching a Drive path or glob strictly inside allowed roots."""\n    if "\\x00" in (spec or ""):\n        raise ValueError("Invalid NUL byte in path")\n    roots = allowed_drive_roots(data_dir, mydrive_root)\n    ignored = skip_dirs if skip_dirs is not None else DEFAULT_SKIP_DIRS\n    raw = (spec or "").strip()\n    is_file_scheme = raw.startswith("file://")\n    for prefix in ("drive://", "gdrive://", "file://"):\n        if raw.startswith(prefix):\n            raw = raw[len(prefix):]\n            break\n\n    has_glob = any(ch in raw for ch in ("*", "?", "["))\n    matched_paths: list[Path] = []\n\n    if has_glob:\n        if is_file_scheme:\n            bases = [Path("/" + raw.lstrip("/"))]\n        else:\n            clean = raw.lstrip("/")\n            if raw.startswith("/") and not clean.startswith(("content/drive/", "drive/", "MyDrive/")):\n                bases = [Path("/" + clean)]\n            elif clean.startswith("content/drive/"):\n                bases = [Path("/") / clean]\n            elif clean.startswith("MyDrive/"):\n                bases = [drive_root / clean]\n            else:\n                bases = []\n                if default_subdir:\n                    bases.append(data_dir / default_subdir / clean)\n                bases.append(data_dir / clean)\n                if mydrive_root.is_dir():\n                    bases.append(mydrive_root / clean)\n\n        # Ensure static prefix before wildcard does not escape allowed roots\n        for pattern_path in bases:\n            prefix_parts: list[str] = []\n            for part in pattern_path.parts:\n                if any(ch in part for ch in ("*", "?", "[")):\n                    break\n                prefix_parts.append(part)\n            if prefix_parts:\n                _ensure_within_roots(Path(*prefix_parts), roots, spec)\n\n        seen: set[str] = set()\n        for pattern_path in bases:\n            for hit in sorted(glob_mod.glob(str(pattern_path), recursive=True)):\n                hp = Path(hit).resolve()\n                if not any(hp.is_relative_to(r) for r in roots):\n                    continue\n                if hp.is_file() and hp.suffix.lower() in AUDIO_EXTS and str(hp) not in seen:\n                    seen.add(str(hp))\n                    matched_paths.append(hp)\n    else:\n        target = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            default_subdir=default_subdir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if target.is_file():\n            if target.suffix.lower() in AUDIO_EXTS:\n                matched_paths.append(target)\n        elif target.is_dir():\n            if not recursive:\n                for hp in sorted(target.iterdir()):\n                    resolved = hp.resolve()\n                    if any(resolved.is_relative_to(r) for r in roots) and resolved.is_file() and resolved.suffix.lower() in AUDIO_EXTS:\n                        matched_paths.append(resolved)\n            else:\n                for root_dir, dirnames, filenames in os.walk(target):\n                    dirnames[:] = sorted(d for d in dirnames if d not in ignored and not d.startswith("."))\n                    for fn in sorted(filenames):\n                        hp = (Path(root_dir) / fn).resolve()\n                        if any(hp.is_relative_to(r) for r in roots) and hp.is_file() and hp.suffix.lower() in AUDIO_EXTS:\n                            matched_paths.append(hp)\n\n    items: list[dict[str, Any]] = []\n    for p in matched_paths:\n        st = p.stat()\n        items.append({\n            "drive_path": str(p),\n            "rel_path": rel_to_mydrive(p, mydrive_root),\n            "name": p.name,\n            "size_bytes": st.st_size,\n            "size_mb": round(st.st_size / 1e6, 2),\n            "mtime": int(st.st_mtime),\n        })\n    return items\n\n\ndef audio_path_to_wav_data_url(\n    audio_path: Path,\n    max_duration_sec: float = 24.0,\n    sample_rate: int = 24000,\n    timeout: int = 60,\n) -> str:\n    """Decode a Drive audio file to a mono PCM16 WAV data URL with bounded ffmpeg timeout."""\n    if not audio_path.is_file():\n        raise FileNotFoundError(f"Audio file not found: {audio_path}")\n    cmd = [\n        "ffmpeg", "-nostdin", "-v", "error",\n        "-t", f"{max_duration_sec:.1f}",\n        "-i", str(audio_path),\n        "-vn", "-ac", "1", "-ar", str(sample_rate), "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n    ]\n    try:\n        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, check=False)\n    except OSError as os_err:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {os_err}") from os_err\n    if res.returncode != 0 or not res.stdout:\n        if audio_path.suffix.lower() == ".wav":\n            b64 = base64.b64encode(audio_path.read_bytes()).decode("ascii")\n            return f"data:audio/wav;base64,{b64}"\n        err = res.stderr.decode("utf-8", errors="replace").strip()\n        raise RuntimeError(f"ffmpeg failed decoding {audio_path.name}: {err}")\n    b64 = base64.b64encode(res.stdout).decode("ascii")\n    return f"data:audio/wav;base64,{b64}"\n\n\ndef save_output_audio_to_drive(\n    audio_bytes: bytes,\n    spec: str,\n    *,\n    data_dir: Path,\n    output_dir: Path,\n    prefix: str,\n    fmt: str = "wav",\n    mydrive_root: Path = Path("/content/drive/MyDrive"),\n    drive_root: Path = Path("/content/drive"),\n) -> str:\n    """Persist synthesized audio bytes safely inside Drive or service output directory."""\n    output_dir.mkdir(parents=True, exist_ok=True)\n    ext = "." + re.sub(r"[^a-z0-9]", "", (fmt or "wav").lower())\n    if not ext or ext == ".":\n        ext = ".wav"\n    if spec and spec not in ("auto", "true", "True", "1"):\n        dest = resolve_drive_path(\n            spec,\n            data_dir=data_dir,\n            mydrive_root=mydrive_root,\n            drive_root=drive_root,\n        )\n        if dest.is_dir() or spec.endswith("/"):\n            dest.mkdir(parents=True, exist_ok=True)\n            dest = dest / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n        else:\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            if not dest.suffix:\n                dest = dest.with_suffix(ext)\n    else:\n        dest = output_dir / f"{prefix}-{time.strftime(\'%Y%m%d-%H%M%S\')}{ext}"\n    dest.write_bytes(audio_bytes)\n    return str(dest)\n\n\ndef tail_service_log(\n    log_file: Path = Path("/content/ct/service.log"),\n    lines: int = 120,\n) -> dict[str, Any]:\n    n_lines = max(1, min(int(lines), 2000))\n    entries = log_file.read_text(encoding="utf-8", errors="replace").splitlines()[-n_lines:] if log_file.is_file() else []\n    return {"lines": len(entries), "log": "\\n".join(entries)}\n', encoding="utf-8")
(QT_WORK / "ct_qwen3_tts_middleware.py").write_text('"""Authenticated ASGI middleware and multi-task routing for Qwen3-TTS."""\nfrom __future__ import annotations\n\nimport asyncio\nimport base64\nimport json\nimport os\nimport shutil\nimport subprocess\nimport time\nimport urllib.error\nimport urllib.request\nfrom http import HTTPStatus\nfrom pathlib import Path\nfrom typing import Any\nfrom urllib.parse import parse_qs\n\nfrom colab_asgi_common import (\n    audio_path_to_wav_data_url,\n    build_multipart_form,\n    list_drive_audio_files,\n    resolve_drive_path,\n    sanitize_header_param,\n    save_output_audio_to_drive,\n    tail_service_log,\n    validate_voice_name,\n    verify_bearer_auth,\n)\n\ntry:\n    from starlette.datastructures import Headers\n    from starlette.responses import JSONResponse, Response\n    from starlette.types import ASGIApp, Message, Receive, Scope, Send\nexcept ImportError:  # pragma: no cover - local unit-test fallback without starlette\n    Headers = dict  # type: ignore[assignment,misc]\n    JSONResponse = Response = None  # type: ignore[assignment,misc]\n    ASGIApp = Message = Receive = Scope = Send = Any  # type: ignore[assignment,misc]\n\nAPI_TOKEN = os.environ.get("CT_API_TOKEN", "")\nSERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8000"))\nAUX_PORT = int(os.environ.get("CT_QWEN3_TTS_AUX_PORT", "8001"))\nDATA_DIR = Path(os.environ.get("CT_DATA_DIR", "/content/drive/MyDrive/colab-llm/qwen3-tts"))\nSPEAKERS_DIR = Path(os.environ.get("SPEAKER_SAMPLES_DIR", "/content/qwen3-tts/speakers"))\nDEFAULT_REF_WAV = Path(os.environ.get("CT_DEFAULT_REF_WAV", "/content/qwen3-tts/default_ref.wav"))\nDEFAULT_REF_TEXT = os.environ.get(\n    "CT_DEFAULT_REF_TEXT",\n    "Okay. Yeah. I resent you. I love you. I respect you. But you know what? You blew it! And thanks to you.",\n)\nPRIMARY_MODEL_REPO = os.environ.get("CT_QWEN3_TTS_MODEL", "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice")\nDEFAULT_SPEAKER = os.environ.get("CT_QWEN3_DEFAULT_SPEAKER", "vivian").strip().lower()\n\nDRIVE_ROOT = Path("/content/drive")\nMYDRIVE_ROOT = Path("/content/drive/MyDrive")\nDRIVE_VOICES_DIR = DATA_DIR / "voices"\nDRIVE_OUTPUT_DIR = DATA_DIR / "outputs"\nCUSTOM_VOICES_META_FILE = SPEAKERS_DIR / "custom_voices_meta.json"\n\nBUILTIN_SPEAKERS = {\n    "vivian": {"description": "Bright, slightly edgy young female voice", "native_language": "Chinese"},\n    "serena": {"description": "Warm, gentle young female voice", "native_language": "Chinese"},\n    "uncle_fu": {"description": "Seasoned male voice with a low, mellow timbre", "native_language": "Chinese"},\n    "dylan": {"description": "Youthful Beijing male voice with a clear, natural timbre", "native_language": "Chinese (Beijing Dialect)"},\n    "eric": {"description": "Lively Chengdu male voice with a slightly husky brightness", "native_language": "Chinese (Sichuan Dialect)"},\n    "ryan": {"description": "Dynamic male voice with strong rhythmic drive", "native_language": "English"},\n    "aiden": {"description": "Sunny American male voice with a clear midrange", "native_language": "English"},\n    "ono_anna": {"description": "Playful Japanese female voice with a light, nimble timbre", "native_language": "Japanese"},\n    "sohee": {"description": "Warm Korean female voice with rich emotion", "native_language": "Korean"},\n}\n\nLANG_MAP = {\n    "auto": "Auto",\n    "zhen": "Auto",\n    "zh": "Chinese",\n    "cn": "Chinese",\n    "chinese": "Chinese",\n    "mandarin": "Chinese",\n    "en": "English",\n    "english": "English",\n    "ja": "Japanese",\n    "jp": "Japanese",\n    "japanese": "Japanese",\n    "ko": "Korean",\n    "kr": "Korean",\n    "korean": "Korean",\n    "de": "German",\n    "german": "German",\n    "fr": "French",\n    "french": "French",\n    "ru": "Russian",\n    "russian": "Russian",\n    "pt": "Portuguese",\n    "portuguese": "Portuguese",\n    "es": "Spanish",\n    "spanish": "Spanish",\n    "it": "Italian",\n    "italian": "Italian",\n}\n\n\ndef _detect_primary_task_type() -> str:\n    low = PRIMARY_MODEL_REPO.lower()\n    if "voicedesign" in low or "voice-design" in low or "voice_design" in low:\n        return "VoiceDesign"\n    if "base" in low:\n        return "Base"\n    return "CustomVoice"\n\n\nPRIMARY_TASK_TYPE = _detect_primary_task_type()\n\n\ndef _verify_auth(headers: Any) -> bool:\n    return verify_bearer_auth(headers, API_TOKEN)\n\n\ndef _resolve_path(raw_spec: str, default_subdir: str | None = "voices") -> Path:\n    return resolve_drive_path(\n        raw_spec,\n        data_dir=DATA_DIR,\n        default_subdir=default_subdir,\n        mydrive_root=MYDRIVE_ROOT,\n        drive_root=DRIVE_ROOT,\n    )\n\n\ndef _load_voices_meta() -> dict[str, dict[str, Any]]:\n    for p in (CUSTOM_VOICES_META_FILE, DRIVE_VOICES_DIR / "custom_voices_meta.json"):\n        if p.is_file():\n            try:\n                return json.loads(p.read_text(encoding="utf-8"))\n            except Exception:\n                pass\n    return {}\n\n\ndef _save_voices_meta(meta: dict[str, dict[str, Any]]) -> None:\n    try:\n        SPEAKERS_DIR.mkdir(parents=True, exist_ok=True)\n        DRIVE_VOICES_DIR.mkdir(parents=True, exist_ok=True)\n        raw = json.dumps(meta, indent=2, ensure_ascii=False)\n        CUSTOM_VOICES_META_FILE.write_text(raw, encoding="utf-8")\n        (DRIVE_VOICES_DIR / "custom_voices_meta.json").write_text(raw, encoding="utf-8")\n    except Exception:\n        pass\n\n\ndef _sync_speakers_to_drive() -> None:\n    try:\n        DRIVE_VOICES_DIR.mkdir(parents=True, exist_ok=True)\n        if SPEAKERS_DIR.is_dir():\n            for sf in SPEAKERS_DIR.iterdir():\n                if sf.is_file():\n                    dst = DRIVE_VOICES_DIR / sf.name\n                    if not dst.is_file() or sf.stat().st_mtime > dst.stat().st_mtime:\n                        shutil.copy2(sf, dst)\n    except Exception:\n        pass\n\n\ndef _register_custom_voice_bytes(\n    name: str,\n    wav_bytes: bytes,\n    ref_text: str | None = None,\n    speaker_desc: str | None = None,\n    consent: str = "user-consent",\n) -> tuple[int, dict[str, Any]]:\n    vname = validate_voice_name(name)\n    SPEAKERS_DIR.mkdir(parents=True, exist_ok=True)\n    DRIVE_VOICES_DIR.mkdir(parents=True, exist_ok=True)\n    wav_file = SPEAKERS_DIR / f"{vname}.wav"\n    wav_file.write_bytes(wav_bytes)\n    shutil.copy2(wav_file, DRIVE_VOICES_DIR / f"{vname}.wav")\n\n    meta = _load_voices_meta()\n    meta[vname] = {\n        "id": vname,\n        "wav_path": str(wav_file),\n        "drive_wav_path": str(DRIVE_VOICES_DIR / f"{vname}.wav"),\n        "ref_text": ref_text,\n        "speaker_description": speaker_desc or "Custom cloned voice for Qwen3-TTS",\n        "file_size": len(wav_bytes),\n        "mime_type": "audio/wav",\n        "created_at": int(time.time()),\n    }\n    _save_voices_meta(meta)\n\n    fields: list[tuple[str, str]] = [("name", vname), ("consent", consent)]\n    if ref_text:\n        fields.append(("ref_text", str(ref_text)))\n    if speaker_desc:\n        fields.append(("speaker_description", str(speaker_desc)))\n    content_type, body_bytes = build_multipart_form(\n        fields,\n        [("audio_sample", f"{vname}.wav", "audio/wav", wav_bytes)],\n    )\n    req = urllib.request.Request(\n        f"http://127.0.0.1:{SERVICE_PORT}/v1/audio/voices",\n        data=body_bytes,\n        headers={\n            "Content-Type": content_type,\n            "Authorization": f"Bearer {API_TOKEN}",\n        },\n        method="POST",\n    )\n    try:\n        with urllib.request.urlopen(req, timeout=60) as resp:\n            out = json.loads(resp.read().decode("utf-8", errors="replace"))\n            _sync_speakers_to_drive()\n            return resp.status, out\n    except Exception:\n        return 200, {"voice": meta[vname]}\n\n\ndef _upload_drive_voice_sync(body: dict[str, Any]) -> tuple[int, dict[str, Any]]:\n    raw_name = str(body.get("name") or "").strip()\n    raw_path = str(body.get("drive_path") or body.get("audio") or body.get("path") or "").strip()\n    audio_b64 = str(body.get("audio_base64") or "").strip()\n    consent = str(body.get("consent") or "user-consent").strip()\n    ref_text = body.get("ref_text")\n    speaker_desc = body.get("speaker_description")\n    if not raw_name or (not raw_path and not audio_b64):\n        return 400, {"error": "\'name\' and either \'drive_path\' or \'audio_base64\' are required"}\n    try:\n        name = validate_voice_name(raw_name)\n    except ValueError as err:\n        return 400, {"error": str(err)}\n\n    if audio_b64:\n        if "," in audio_b64 and audio_b64.startswith("data:"):\n            audio_b64 = audio_b64.split(",", 1)[1]\n        raw_bytes = base64.b64decode(audio_b64)\n        cmd = [\n            "ffmpeg", "-nostdin", "-v", "error", "-t", "24.0",\n            "-i", "pipe:0", "-vn", "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n        ]\n        try:\n            res = subprocess.run(cmd, input=raw_bytes, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=60, check=False)\n            wav_bytes = res.stdout if (res.returncode == 0 and res.stdout) else raw_bytes\n        except OSError:\n            wav_bytes = raw_bytes\n    else:\n        try:\n            ap = _resolve_path(raw_path)\n        except ValueError as err:\n            return 400, {"error": str(err)}\n        if not ap.is_file():\n            return 404, {"error": f"Audio file not found on Google Drive: {raw_path} (resolved to {ap})"}\n        cmd = [\n            "ffmpeg", "-nostdin", "-v", "error", "-t", "24.0",\n            "-i", str(ap), "-vn", "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", "-f", "wav", "pipe:1",\n        ]\n        try:\n            res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=60, check=False)\n            wav_bytes = res.stdout if (res.returncode == 0 and res.stdout) else ap.read_bytes()\n        except OSError:\n            wav_bytes = ap.read_bytes()\n\n    return _register_custom_voice_bytes(name, wav_bytes, ref_text=ref_text, speaker_desc=speaker_desc, consent=consent)\n\n\ndef _prepare_speech_request(raw_body: bytes) -> tuple[bytes, dict[str, Any], str, str | None]:\n    payload = json.loads(raw_body) if raw_body else {}\n    if not isinstance(payload, dict):\n        payload = {}\n\n    save_drive_spec: str | None = None\n    extras = payload.get("extra_params")\n    if not isinstance(extras, dict):\n        extras = {}\n\n    for save_key in ("save_to_drive", "save_drive_path"):\n        val = payload.pop(save_key, None)\n        if val is None and save_key in extras:\n            val = extras.pop(save_key, None)\n        if val:\n            save_drive_spec = str(val) if not isinstance(val, bool) else "auto"\n\n    for k in ("language", "lang", "task_type", "instructions", "instruct", "ref_audio", "ref_text", "x_vector_only_mode"):\n        if k in extras and k not in payload:\n            payload[k] = extras.pop(k)\n\n    if "instruct" in payload and not payload.get("instructions"):\n        payload["instructions"] = payload.pop("instruct")\n    else:\n        payload.pop("instruct", None)\n\n    raw_lang = str(payload.pop("lang", None) or payload.get("language") or "Auto").strip()\n    payload["language"] = LANG_MAP.get(raw_lang.lower(), raw_lang.title() if raw_lang else "Auto")\n\n    # Resolve ref_audio if present (reject arbitrary external file:// outside Drive)\n    ref_audio = payload.get("ref_audio")\n    if isinstance(ref_audio, str) and ref_audio.strip():\n        ra = ref_audio.strip()\n        if not ra.startswith(("http://", "https://", "data:")):\n            ap = _resolve_path(ra)\n            payload["ref_audio"] = audio_path_to_wav_data_url(ap)\n\n    meta = _load_voices_meta()\n    raw_voice = str(payload.get("voice") or payload.get("speaker") or "").strip().lower()\n    payload.pop("speaker", None)\n    is_custom_uploaded_voice = bool(raw_voice and raw_voice not in BUILTIN_SPEAKERS and raw_voice not in ("default", "alloy"))\n    if raw_voice in meta and not payload.get("ref_audio"):\n        vinfo = meta[raw_voice]\n        wpath = Path(vinfo.get("wav_path") or "")\n        if not wpath.is_file():\n            wpath = Path(vinfo.get("drive_wav_path") or "")\n        if wpath.is_file():\n            payload["ref_audio"] = f"file://{wpath}"\n            if vinfo.get("ref_text") and not payload.get("ref_text"):\n                payload["ref_text"] = vinfo["ref_text"]\n            is_custom_uploaded_voice = True\n\n    req_task = str(payload.get("task_type") or "").strip()\n    if req_task.lower() in ("voicedesign", "voice_design", "voice-design"):\n        task_type = "VoiceDesign"\n    elif req_task.lower() in ("base", "voiceclone", "voice_clone", "voice-clone") or payload.get("ref_audio") or is_custom_uploaded_voice:\n        task_type = "Base"\n    elif req_task.lower() in ("customvoice", "custom_voice", "custom-voice"):\n        task_type = "CustomVoice"\n    else:\n        task_type = PRIMARY_TASK_TYPE\n\n    payload["task_type"] = task_type\n\n    if task_type == "CustomVoice":\n        if not raw_voice or raw_voice in ("default", "alloy"):\n            raw_voice = "ryan" if payload.get("language") == "English" else DEFAULT_SPEAKER\n        payload["voice"] = raw_voice\n        payload.pop("ref_audio", None)\n        payload.pop("ref_text", None)\n        payload.pop("x_vector_only_mode", None)\n    elif task_type == "VoiceDesign":\n        payload.pop("voice", None)\n        payload.pop("ref_audio", None)\n        payload.pop("ref_text", None)\n        payload.pop("x_vector_only_mode", None)\n        if not payload.get("instructions"):\n            payload["instructions"] = "Natural, expressive human voice."\n    elif task_type == "Base":\n        if not payload.get("ref_audio") and (not raw_voice or raw_voice in ("default", "alloy") or raw_voice in BUILTIN_SPEAKERS):\n            payload["ref_audio"] = f"file://{DEFAULT_REF_WAV}"\n            if not payload.get("ref_text") and payload.get("x_vector_only_mode") is None:\n                payload["ref_text"] = DEFAULT_REF_TEXT\n            payload.pop("voice", None)\n        elif payload.get("ref_audio"):\n            payload.pop("voice", None)\n        if payload.get("x_vector_only_mode") is None:\n            payload["x_vector_only_mode"] = not bool(payload.get("ref_text") and str(payload.get("ref_text")).strip())\n\n    if extras:\n        payload["extra_params"] = extras\n    else:\n        payload.pop("extra_params", None)\n\n    return json.dumps(payload, ensure_ascii=False).encode("utf-8"), payload, task_type, save_drive_spec\n\n\ndef _call_aux_worker_sync(payload: dict[str, Any]) -> tuple[int, bytes, dict[str, str]]:\n    req = urllib.request.Request(\n        f"http://127.0.0.1:{AUX_PORT}/synthesize",\n        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),\n        headers={"Content-Type": "application/json"},\n        method="POST",\n    )\n    try:\n        with urllib.request.urlopen(req, timeout=300) as resp:\n            headers = {\n                "content-type": resp.headers.get("Content-Type") or "audio/wav",\n                "x-audio-sample-rate": resp.headers.get("X-Audio-Sample-Rate") or "24000",\n                "x-qwen3-tts-engine": resp.headers.get("X-Qwen3-TTS-Engine") or "aux",\n            }\n            return resp.status, resp.read(), headers\n    except urllib.error.HTTPError as err:\n        return err.code, err.read(), {"content-type": "application/json"}\n\n\ndef _design_and_register_voice_sync(body: dict[str, Any]) -> tuple[int, dict[str, Any]]:\n    instruct = str(body.get("instructions") or body.get("instruct") or "").strip()\n    sample_text = str(body.get("sample_text") or body.get("text") or body.get("input") or "").strip()\n    language = LANG_MAP.get(str(body.get("language") or body.get("lang") or "Auto").strip().lower(), "Auto")\n    raw_voice_name = str(body.get("name") or "").strip()\n    voice_name = ""\n    if raw_voice_name:\n        try:\n            voice_name = validate_voice_name(raw_voice_name)\n        except ValueError as err:\n            return 400, {"error": str(err)}\n    if not instruct:\n        return 400, {"error": "\'instructions\' (voice persona description) is required for Voice Design"}\n    if not sample_text:\n        sample_text = (\n            "Hello! This is a sample of my designed voice, ready for speech synthesis."\n            if language == "English"\n            else "Hello! This is a sample of the custom voice generated from your description."\n        )\n\n    status, audio_bytes, headers = _call_aux_worker_sync({\n        "task_type": "VoiceDesign",\n        "input": sample_text,\n        "language": language,\n        "instructions": instruct,\n        "response_format": "wav",\n    })\n    if status != 200:\n        try:\n            return status, json.loads(audio_bytes.decode("utf-8", errors="replace"))\n        except Exception:\n            return status, {"error": audio_bytes.decode("utf-8", errors="replace")}\n\n    result: dict[str, Any] = {\n        "status": "ok",\n        "task_type": "VoiceDesign",\n        "instructions": instruct,\n        "sample_text": sample_text,\n        "language": language,\n        "sample_rate": int(headers.get("x-audio-sample-rate") or 24000),\n        "audio_base64": base64.b64encode(audio_bytes).decode("ascii"),\n    }\n    if voice_name:\n        _, reg_out = _register_custom_voice_bytes(\n            voice_name,\n            audio_bytes,\n            ref_text=sample_text,\n            speaker_desc=f"VoiceDesign: {instruct}",\n            consent="voice-design",\n        )\n        result["registered_voice"] = voice_name\n        result["registration"] = reg_out\n    if body.get("save_to_drive"):\n        try:\n            result["saved_drive_path"] = save_output_audio_to_drive(\n                audio_bytes,\n                str(body.get("save_to_drive")),\n                data_dir=DATA_DIR,\n                output_dir=DRIVE_OUTPUT_DIR,\n                prefix="qwen3-tts",\n                fmt="wav",\n                mydrive_root=MYDRIVE_ROOT,\n                drive_root=DRIVE_ROOT,\n            )\n        except ValueError as err:\n            return 400, {"error": str(err)}\n    return 200, result\n\n\nclass Qwen3TTSMiddleware:\n    def __init__(self, app: ASGIApp) -> None:\n        self.app = app\n\n    async def __call__(self, scope: Scope, receive: Receive, send: Send) -> None:\n        if scope["type"] not in ("http", "websocket"):\n            await self.app(scope, receive, send)\n            return\n        if scope.get("method") == "OPTIONS":\n            await self.app(scope, receive, send)\n            return\n\n        root_path = scope.get("root_path", "")\n        path = scope["path"].removeprefix(root_path)\n        headers = Headers(scope=scope)\n\n        if not _verify_auth(headers):\n            resp = JSONResponse(\n                {"error": {"message": "Unauthorized: invalid or missing Bearer token", "type": "AuthenticationError", "code": 401}},\n                status_code=HTTPStatus.UNAUTHORIZED,\n            )\n            await resp(scope, receive, send)\n            return\n\n        if scope["type"] == "http":\n            method = scope.get("method", "GET")\n\n            if method == "GET" and path == "/api/ct/info":\n                meta = _load_voices_meta()\n                resp = JSONResponse({\n                    "service": "qwen3-tts",\n                    "primary_model": PRIMARY_MODEL_REPO,\n                    "primary_task_type": PRIMARY_TASK_TYPE,\n                    "supported_tasks": ["CustomVoice", "VoiceDesign", "Base"],\n                    "builtin_speakers": BUILTIN_SPEAKERS,\n                    "custom_voices": list(meta.keys()),\n                    "languages": sorted(set(LANG_MAP.values())),\n                    "default_speaker": DEFAULT_SPEAKER,\n                    "drive_voices_dir": str(DRIVE_VOICES_DIR),\n                    "drive_output_dir": str(DRIVE_OUTPUT_DIR),\n                })\n                await resp(scope, receive, send)\n                return\n\n            if method == "GET" and path == "/api/ct/logs":\n                qs = parse_qs((scope.get("query_string") or b"").decode("utf-8", errors="replace"))\n                try:\n                    n_lines = int((qs.get("lines") or ["120"])[0])\n                except ValueError:\n                    n_lines = 120\n                resp = JSONResponse(tail_service_log(lines=n_lines))\n                await resp(scope, receive, send)\n                return\n\n            if method == "GET" and path == "/api/ct/drive-list":\n                qs = parse_qs((scope.get("query_string") or b"").decode("utf-8", errors="replace"))\n                spec = (qs.get("path") or [""])[0]\n                recursive = (qs.get("recursive") or ["1"])[0] not in ("0", "false", "False")\n                try:\n                    files = await asyncio.to_thread(\n                        list_drive_audio_files,\n                        spec,\n                        data_dir=DATA_DIR,\n                        default_subdir="voices",\n                        recursive=recursive,\n                        mydrive_root=MYDRIVE_ROOT,\n                        drive_root=DRIVE_ROOT,\n                    )\n                    resp = JSONResponse({\n                        "drive_mounted": MYDRIVE_ROOT.is_dir(),\n                        "query": spec,\n                        "count": len(files),\n                        "files": files,\n                    })\n                except Exception as e:\n                    resp = JSONResponse({"error": str(e)}, status_code=400)\n                await resp(scope, receive, send)\n                return\n\n            if method == "POST" and path == "/api/ct/drive-upload-voice":\n                chunks: list[bytes] = []\n                while True:\n                    msg = await receive()\n                    if msg["type"] == "http.request":\n                        chunks.append(msg.get("body", b""))\n                        if not msg.get("more_body", False):\n                            break\n                    elif msg["type"] == "http.disconnect":\n                        return\n                try:\n                    body = json.loads(b"".join(chunks))\n                    code, out = await asyncio.to_thread(_upload_drive_voice_sync, body)\n                    resp = JSONResponse(out, status_code=code)\n                except ValueError as e:\n                    resp = JSONResponse({"error": str(e)}, status_code=400)\n                except Exception as e:\n                    resp = JSONResponse({"error": str(e)}, status_code=500)\n                await resp(scope, receive, send)\n                return\n\n            if method == "POST" and path == "/api/ct/design-voice":\n                chunks = []\n                while True:\n                    msg = await receive()\n                    if msg["type"] == "http.request":\n                        chunks.append(msg.get("body", b""))\n                        if not msg.get("more_body", False):\n                            break\n                    elif msg["type"] == "http.disconnect":\n                        return\n                try:\n                    body = json.loads(b"".join(chunks))\n                    code, out = await asyncio.to_thread(_design_and_register_voice_sync, body)\n                    resp = JSONResponse(out, status_code=code)\n                except ValueError as e:\n                    resp = JSONResponse({"error": str(e)}, status_code=400)\n                except Exception as e:\n                    resp = JSONResponse({"error": str(e)}, status_code=500)\n                await resp(scope, receive, send)\n                return\n\n            if method == "GET" and path == "/v1/audio/voices":\n                meta = _load_voices_meta()\n                all_voices = sorted(set(BUILTIN_SPEAKERS.keys()) | set(meta.keys()))\n                resp = JSONResponse({\n                    "voices": all_voices,\n                    "builtin_speakers": BUILTIN_SPEAKERS,\n                    "uploaded_voices": list(meta.values()),\n                })\n                await resp(scope, receive, send)\n                return\n\n            if method == "POST" and path == "/v1/audio/speech":\n                chunks = []\n                while True:\n                    msg = await receive()\n                    if msg["type"] == "http.request":\n                        chunks.append(msg.get("body", b""))\n                        if not msg.get("more_body", False):\n                            break\n                    elif msg["type"] == "http.disconnect":\n                        return\n                raw_body = b"".join(chunks)\n                try:\n                    rewritten_body, payload_dict, task_type, save_drive_spec = await asyncio.to_thread(\n                        _prepare_speech_request, raw_body\n                    )\n                except Exception as e:\n                    resp = JSONResponse(\n                        {"error": {"message": str(e), "type": "BadRequestError", "code": 400}},\n                        status_code=400,\n                    )\n                    await resp(scope, receive, send)\n                    return\n\n                header_save = headers.get("x-save-drive")\n                if header_save and not save_drive_spec:\n                    save_drive_spec = header_save\n\n                if task_type != PRIMARY_TASK_TYPE:\n                    status, audio_bytes, aux_headers = await asyncio.to_thread(_call_aux_worker_sync, payload_dict)\n                    if status == 200 and save_drive_spec:\n                        try:\n                            saved_path = await asyncio.to_thread(\n                                save_output_audio_to_drive,\n                                audio_bytes,\n                                save_drive_spec,\n                                data_dir=DATA_DIR,\n                                output_dir=DRIVE_OUTPUT_DIR,\n                                prefix="qwen3-tts",\n                                fmt=str(payload_dict.get("response_format") or "wav"),\n                                mydrive_root=MYDRIVE_ROOT,\n                                drive_root=DRIVE_ROOT,\n                            )\n                        except ValueError as e:\n                            resp = JSONResponse({"error": str(e)}, status_code=400)\n                            await resp(scope, receive, send)\n                            return\n                        aux_headers["x-saved-drive-path"] = sanitize_header_param(saved_path)\n                    resp = Response(content=audio_bytes, status_code=status, headers=aux_headers)\n                    await resp(scope, receive, send)\n                    return\n\n                body_sent = False\n\n                async def replay_receive() -> Message:\n                    nonlocal body_sent\n                    if not body_sent:\n                        body_sent = True\n                        return {"type": "http.request", "body": rewritten_body, "more_body": False}\n                    return await receive()\n\n                if not save_drive_spec:\n                    await self.app(scope, replay_receive, send)\n                    return\n\n                resp_status = 200\n                resp_headers: list[tuple[bytes, bytes]] = []\n                resp_chunks: list[bytes] = []\n                start_msg: Message | None = None\n\n                async def tee_send(message: Message) -> None:\n                    nonlocal resp_status, resp_headers, start_msg\n                    if message["type"] == "http.response.start":\n                        resp_status = message.get("status", 200)\n                        resp_headers = list(message.get("headers", []))\n                        start_msg = message\n                        if resp_status != 200:\n                            await send(message)\n                    elif message["type"] == "http.response.body":\n                        chunk = message.get("body", b"")\n                        more = message.get("more_body", False)\n                        if resp_status != 200:\n                            await send(message)\n                            return\n                        if chunk:\n                            resp_chunks.append(chunk)\n                        if not more and start_msg is not None:\n                            full_audio = b"".join(resp_chunks)\n                            saved_path = await asyncio.to_thread(\n                                save_output_audio_to_drive,\n                                full_audio,\n                                save_drive_spec,\n                                data_dir=DATA_DIR,\n                                output_dir=DRIVE_OUTPUT_DIR,\n                                prefix="qwen3-tts",\n                                fmt=str(payload_dict.get("response_format") or "wav"),\n                                mydrive_root=MYDRIVE_ROOT,\n                                drive_root=DRIVE_ROOT,\n                            )\n                            safe_hdr = sanitize_header_param(saved_path).encode("utf-8")\n                            new_headers = [\n                                (k, v) for (k, v) in resp_headers if k.lower() != b"x-saved-drive-path"\n                            ]\n                            new_headers.append((b"x-saved-drive-path", safe_hdr))\n                            start_msg["headers"] = new_headers\n                            await send(start_msg)\n                            await send({"type": "http.response.body", "body": full_audio, "more_body": False})\n                    else:\n                        await send(message)\n\n                await self.app(scope, replay_receive, tee_send)\n                return\n\n            if method in ("POST", "DELETE") and path.startswith("/v1/audio/voices"):\n                if method == "DELETE":\n                    raw_vname = path.rstrip("/").split("/")[-1]\n                    try:\n                        vname = validate_voice_name(raw_vname)\n                    except ValueError as err:\n                        resp = JSONResponse({"error": str(err)}, status_code=400)\n                        await resp(scope, receive, send)\n                        return\n                    meta = _load_voices_meta()\n                    if vname in meta:\n                        meta.pop(vname, None)\n                        _save_voices_meta(meta)\n                        for d in (SPEAKERS_DIR, DRIVE_VOICES_DIR):\n                            (d / f"{vname}.wav").unlink(missing_ok=True)\n                await self.app(scope, receive, send)\n                await asyncio.to_thread(_sync_speakers_to_drive)\n                return\n\n        await self.app(scope, receive, send)\n', encoding="utf-8")

VLLM_ENV = {
    "PATH": f"{VENV_DIR}/bin:/usr/local/cuda/bin:{os.environ.get('PATH', '')}",
    "PYTHONPATH": f"{QT_WORK}:{os.environ.get('PYTHONPATH', '')}",
    "VIRTUAL_ENV": str(VENV_DIR),
    "FLASHINFER_DISABLE_VERSION_CHECK": "1",
    "FLASHINFER_WORKSPACE_BASE": str(FLASHINFER_CACHE_DIR),
    "TRITON_CACHE_DIR": str(TRITON_CACHE_DIR),
    "VLLM_CACHE_ROOT": str(VLLM_CACHE_DIR),
    "TORCHINDUCTOR_CACHE_DIR": str(VLLM_CACHE_DIR / "inductor"),
    "SPEAKER_SAMPLES_DIR": str(SPEAKERS_DIR),
    "CT_API_TOKEN": API_TOKEN,
    "CT_SERVICE_PORT": str(SERVICE_PORT),
    "CT_QWEN3_TTS_AUX_PORT": str(AUX_PORT),
    "CT_DATA_DIR": str(DATA_DIR),
    "CT_MODELS_ROOT": str(MODELS_ROOT),
    "CT_DEFAULT_REF_WAV": str(DEFAULT_REF_WAV),
    "CT_DEFAULT_REF_TEXT": DEFAULT_REF_TEXT,
    "CT_QWEN3_TTS_MODEL": MODEL_REPO,
}

SERVER_ARGS = [
    str(VENV_DIR / "bin" / "vllm"), "serve", str(MODEL_DIR),
    "--omni",
    "--served-model-name", MODEL_REPO, MODEL_ALIAS,
    "--host", "127.0.0.1",
    "--port", str(SERVICE_PORT),
    "--api-key", API_TOKEN,
    "--trust-remote-code",
    "--allowed-local-media-path", str(QT_WORK),
    "--stage-init-timeout", "900",
    "--deploy-config", str(DEPLOY_YAML),
    "--middleware", "ct_qwen3_tts_middleware.Qwen3TTSMiddleware",
]

AUX_ARGS = [
    str(AUX_VENV_DIR / "bin" / "python"),
    str(QT_WORK / "ct_qwen3_aux_worker.py"),
]

def start_aux():
    subprocess.run(["pkill", "-9", "-f", "ct_qwen3_aux_worker.py"], check=False)
    return ct_spawn("aux", AUX_ARGS, env=VLLM_ENV, cwd=str(QT_WORK))

def start_service():
    subprocess.run(["pkill", "-9", "-f", "vllm|EngineCore"], check=False)
    time.sleep(1)
    qlog(f"Starting vLLM-Omni Qwen3-TTS server on 127.0.0.1:{SERVICE_PORT} (model={MODEL_REPO})")
    return ct_spawn("service", SERVER_ARGS, env=VLLM_ENV, cwd=str(QT_WORK))

PROCS["aux"] = (start_aux(), start_aux)
PROCS["service"] = (start_service(), start_service)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)
import platform
CADDY_ARCH = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")

def start_caddy():
    if not os.path.exists("/usr/local/bin/caddy"):
        tag = json.loads(urllib.request.urlopen(
            "https://api.github.com/repos/caddyserver/caddy/releases/latest", timeout=30).read())["tag_name"]
        tgz = "/tmp/caddy.tgz"
        with urllib.request.urlopen(f"https://github.com/caddyserver/caddy/releases/download/{tag}/"
                                    f"caddy_{tag.lstrip('v')}_linux_{CADDY_ARCH}.tar.gz", timeout=60) as src, \
                open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        subprocess.run(["tar", "-xzf", tgz, "-C", "/usr/local/bin", "caddy"], check=True)
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output(["caddy", "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    open(f"{WORK}/Caddyfile", "w").write(caddyfile)
    subprocess.run(["pkill", "-f", "caddy run"], check=False)
    return ct_spawn("caddy", ["caddy", "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version):
    if destination.exists():
        return str(destination)
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            with tar.extractfile(entry) as source:
                destination.write_bytes(source.read())
            destination.chmod(0o755)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx['WORK'])
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        (work/(name+'.log')).write_text('')
        proc=ctx['ct_spawn'](name,args)
        marker.write_text(json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        ca_path.write_text(os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        config_path.touch(mode=0o600,exist_ok=True)
        config_path.chmod(0o600)
        config_path.write_text(render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        binary=install_frpc(work/('frpc-'+os.environ['CT_FRP_VERSION']),os.environ['CT_FRP_VERSION'])
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        cf_arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine(),'amd64')
        binary=ctx['ct_fetch_binary']('/usr/local/bin/cloudflared',
            f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}')
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see /content/ct/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- wait for 2-stage load, warmup synthesis & sync JIT caches
BROWSER_UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36"

def http_json(url, key=None, payload=None, timeout=30):
    headers = {"Content-Type": "application/json", "User-Agent": BROWSER_UA}
    if key is not None:
        headers["Authorization"] = "Bearer " + key
    req = urllib.request.Request(
        url, headers=headers, data=None if payload is None else json.dumps(payload).encode("utf-8")
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read().decode("utf-8", errors="replace")
            return resp.status, (json.loads(raw) if raw.strip().startswith(("{", "[")) else {})
    except urllib.error.HTTPError as err:
        return err.code, {}

def auth_check(base, key):
    for candidate, expected in ((None, 401), ("invalid-token-probe", 401), (key, 200)):
        code, _ = http_json(base + "/v1/models", candidate)
        if code != expected:
            raise RuntimeError(f"Auth test failed on {base}/v1/models: expected {expected}, got {code}")

LOCAL_BASE = f"http://127.0.0.1:{SERVICE_PORT}"
deadline = time.monotonic() + 2400
last_notice = 0
while time.monotonic() < deadline:
    proc = PROCS["service"][0]
    if proc.poll() is not None:
        raise RuntimeError("vLLM-Omni exited during startup:\n" + ct_tail("service", 80))
    try:
        code, _ = http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)
        if code == 200:
            break
    except Exception:
        pass
    if time.monotonic() - last_notice >= 30:
        gpu_stat = ct_gpu_stat()
        last_line = (ct_tail("service", 3).strip().splitlines() or ["starting..."])[-1]
        qlog(f"Waiting for 2-stage Qwen3-TTS init | GPU: {gpu_stat} | {last_line[:100]}")
        last_notice = time.monotonic()
    time.sleep(5)
else:
    raise RuntimeError("Timed out waiting for vLLM-Omni server to become ready:\n" + ct_tail("service", 80))

auth_check(LOCAL_BASE, API_TOKEN)

def qwen3_healthy():
    if http_json(LOCAL_BASE + "/health", API_TOKEN, timeout=10)[0] != 200:
        return False
    return http_json(f"http://127.0.0.1:{AUX_PORT}/health", timeout=10)[0] == 200

if http_json(f"http://127.0.0.1:{AUX_PORT}/health", timeout=10)[0] != 200:
    qlog("Auxiliary worker not healthy yet:\n" + ct_tail("aux", 20))

# Warmup synthesis (payload matches the primary checkpoint task) to sync JIT caches to Drive
_warm = {"model": MODEL_REPO, "input": "Hello, the Qwen3 TTS speech synthesis service is ready.",
         "language": "Chinese", "response_format": "wav"}
_low_repo = MODEL_REPO.lower()
if "voicedesign" in _low_repo:
    _warm.update(task_type="VoiceDesign", instructions="Natural, warm female narrator.")
elif "base" in _low_repo:
    _warm.update(task_type="Base")
else:
    _warm.update(voice="vivian")
warmup_req = urllib.request.Request(
    LOCAL_BASE + "/v1/audio/speech",
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_TOKEN}"},
    data=json.dumps(_warm).encode("utf-8"),
)
try:
    with urllib.request.urlopen(warmup_req, timeout=180) as wresp:
        wav_bytes = wresp.read()
        qlog(f"Warmup synthesis succeeded ({len(wav_bytes)} bytes WAV)")
except Exception as werr:
    qlog(f"Warmup synthesis warning: {werr}")

for _local_c, _legacy_c in CACHE_PAIRS:
    try:
        save_cache(_local_c, CACHE_ARCHIVES, CACHE_KEY, qlog)
    except (OSError, ValueError) as err:
        qlog(f"Cache archive save warning ({_local_c.name}): {err}")
# User voice samples remain ordinary Drive files, not compiler-cache archives.
subprocess.run(["cp", "-ru", f"{SPEAKERS_DIR}/.", f"{DRIVE_VOICES_DIR}/"], check=False)

ct_public_gate(PUBLIC_URL, lambda: http_json(PUBLIC_URL + "/v1/models", API_TOKEN, timeout=15)[0],
               lambda: auth_check(PUBLIC_URL, API_TOKEN), qlog, timeout=120, interval=8)
qlog(f"Public endpoint verified: {PUBLIC_URL}/v1/models")

print(f"{SERVICE} is up: {PUBLIC_URL}")
print("CT_READY", flush=True)

ct_watchdog(qwen3_healthy, start_service, qlog, dead_markers=('replica-0 is dead',),
            heartbeat=lambda up: qlog(f"heartbeat | up={up}s | GPU: {ct_gpu_stat()}"))
